In [6]:
import os
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl.styles import PatternFill, Font, Alignment

def get_file_info(base_path, version_suffix):
    base_path = Path(base_path)
    data = []
    
    # 해당 폴더 내 모든 .md 파일 검색
    for file_path in base_path.rglob("*.md"):
        # 파일명에서 버전 접미사 제거
        clean_name = file_path.stem.replace(f'_{version_suffix}', '')
        
        # 정보 수집 (파일 시스템 통계)
        stats = file_path.stat()
        mtime = datetime.fromtimestamp(stats.st_mtime).strftime('%Y-%m-%d %H:%M:%S')
        
        # 라인 수 및 글자 수(공백 포함) 계산
        line_count = 0
        char_count = 0
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                line_count = content.count('\n') + (1 if content else 0) # 라인 수
                char_count = len(content) # 글자 수 (공백/엔터 포함)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            line_count = 0
            char_count = 0

        data.append({
            '파일명': clean_name,
            f'크기_{version_suffix}(KB)': round(stats.st_size / 1024, 2),
            f'라인수_{version_suffix}': line_count,
            f'글자수_{version_suffix}': char_count,  # 공백 포함 카운트 추가
            f'수정일시_{version_suffix}': mtime
        })
    
    return pd.DataFrame(data)

In [7]:
v1='v99'
v2='v84'

# 1. 경로 설정
v91_path = "./essay_"+v1
v92_path = "./essay_"+v2

# 2. 각 버전 데이터 수집
df_91 = get_file_info(v91_path, v1)
df_92 = get_file_info(v92_path, v2)

# 3. '파일명'을 기준으로 병합 (Outer Join을 통해 한쪽에만 있는 파일도 표시)
df_compare = pd.merge(df_91, df_92, on='파일명', how='outer')

# 4. 보기 좋게 정렬 (파일명 기준)
df_compare = df_compare.sort_values('파일명').reset_index(drop=True)

# 결과 확인
df_compare

,파일명,크기_v99(KB),라인수_v99,글자수_v99,수정일시_v99,크기_v84(KB),라인수_v84,글자수_v84,수정일시_v84
0,1. 공리,3.75,13.0,1576.0,2026-05-11 20:58:49,3.73,13.0,1573.0,2026-05-10 16:08:19
1,10. 휨,5.11,15.0,2179.0,2026-05-10 14:47:48,5.09,15.0,2179.0,2026-05-10 16:08:17
2,11. 엔트로피,4.64,13.0,1968.0,2026-05-10 15:37:06,4.64,13.0,1968.0,2026-05-10 16:08:18
3,12. 혼돈,4.42,15.0,1895.0,2026-05-10 15:53:07,4.42,15.0,1895.0,2026-05-10 16:08:17
4,13. 양자,4.59,12.0,1932.0,2026-05-11 20:58:49,3.67,12.0,1557.0,2026-05-10 16:08:19
...,...,...,...,...,...,...,...,...,...
72,수식,2.43,170.0,2473.0,2026-05-06 20:22:36,2.43,170.0,2473.0,2026-05-10 16:08:16
73,원자,4.53,13.0,1915.0,2026-05-11 20:58:47,NaN,NaN,NaN,NaN
74,의지,3.58,11.0,1510.0,2026-05-08 20:38:39,3.57,11.0,1510.0,2026-05-10 16:08:26
75,집필 원칙 - v0.9까지,5.85,140.0,2746.0,2026-05-06 20:22:36,5.85,140.0,2746.0,2026-05-10 16:08:16


In [8]:
excel_file = "essay_comparison.xlsx"

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    df_compare.to_excel(writer, index=False, sheet_name='비교결과')
    
    workbook = writer.book
    worksheet = writer.sheets['비교결과']
    
    # 스타일 정의
    latest_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid') # 연한 녹색 (최신)
    header_fill = PatternFill(start_color='DDEBF7', end_color='DDEBF7', fill_type='solid')
    
    # 헤더 스타일
    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')
    
    # 데이터 행 반복
    for row_num in range(2, len(df_compare) + 2):
        val_91 = worksheet.cell(row=row_num, column=4).value
        val_92 = worksheet.cell(row=row_num, column=7).value
        
        try:
            # 문자열을 datetime 객체로 변환하여 비교
            dt_91 = datetime.strptime(val_91, '%Y-%m-%d %H:%M:%S') if val_91 else None
            dt_92 = datetime.strptime(val_92, '%Y-%m-%d %H:%M:%S') if val_92 else None
            
            if dt_91 and dt_92:
                if dt_91 > dt_92:
                    # v91이 더 최신인 경우
                    worksheet.cell(row=row_num, column=4).fill = latest_fill
                elif dt_92 > dt_91:
                    # v92가 더 최신인 경우
                    worksheet.cell(row=row_num, column=7).fill = latest_fill
            elif dt_91: # 한쪽에만 데이터가 있는 경우 (그 자체가 최신)
                worksheet.cell(row=row_num, column=4).fill = latest_fill
            elif dt_92:
                worksheet.cell(row=row_num, column=7).fill = latest_fill
                
        except (ValueError, TypeError):
            # 날짜 형식이 아니거나 데이터가 없는 경우 스킵
            pass

    # 열 너비 자동 조절
    for col in worksheet.columns:
        max_length = 0
        column_letter = col[0].column_letter
        for cell in col:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        worksheet.column_dimensions[column_letter].width = max_length + 2

print(f"✅ 최신 날짜 강조 엑셀 저장 완료: {excel_file}")

✅ 최신 날짜 강조 엑셀 저장 완료: essay_comparison.xlsx


In [ ]:
# docker run -d --gpus all --name syh-model-server \
#   --restart always \
#   -p 0.0.0.0:48000:8000 \
#   -p 0.0.0.0:40022:22 \
#   -p 0.0.0.0:47860:7860 \
#   -p 0.0.0.0:45000:5000 \
#   -v /mnt/d/docker/yhs_dev:/home/yhsong/dev \
#   -v /mnt/d/docker/share:/home/yhsong/share \
#   -v /mnt/d/docker/share/bin:/home/yhsong/bin \
#   -v /mnt/g:/home/yhsong/gdrive \
#   syh-model-server:latest \
#   /bin/bash -c "su - yhsong -c 'jupyter-lab --ip=0.0.0.0 --port=8000 --no-browser --ServerApp.password_required=True --ServerApp.disable_check_xsrf=
# True'"
